In [6]:
#  Fake Job Detection — Interactive Dash Dashboard
import pandas as pd
import numpy as np
from jupyter_dash import JupyterDash
from dash import dcc, html, Input, Output
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (confusion_matrix, roc_curve, auc,
    f1_score, precision_score, recall_score, precision_recall_curve,
    classification_report)
from scipy.sparse import hstack, csr_matrix
import warnings
warnings.filterwarnings('ignore')

# LOAD & PREPARE DATA
df = pd.read_csv("C:\\Users\\AVI SHARMA\\Documents\\fake real job\\fake_job_postings.csv")
df['text'] = (df['title'].fillna('') + ' ' +
              df['description'].fillna('') + ' ' +
              df['requirements'].fillna('') + ' ' +
              df['company_profile'].fillna(''))

df['desc_len'] = df['description'].fillna('').apply(len)
df['req_len']  = df['requirements'].fillna('').apply(len)
df['text_len'] = df['text'].apply(len)
df['label']    = df['fraudulent'].map({0: 'Real', 1: 'Fake'})

y = df['fraudulent'].values

# TF-IDF + FEATURE MATRIX
tfidf = TfidfVectorizer(
    max_features=4000, ngram_range=(1, 2),
    sublinear_tf=True, stop_words='english', min_df=3
)
X_tfidf = tfidf.fit_transform(df['text'])
X_num = df[['telecommuting', 'has_company_logo',
            'has_questions', 'desc_len', 'req_len', 'text_len']
          ].values.astype(float)
X_all = hstack([X_tfidf, csr_matrix(X_num)])

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y, test_size=0.2, random_state=42, stratify=y
)
# TRAIN MODELS
print("Training models... please wait.")

models = {
    'Logistic Regression': LogisticRegression(
        C=1.0, class_weight='balanced', max_iter=300, solver='saga'),
    'Random Forest': RandomForestClassifier(
        n_estimators=80, max_depth=12, class_weight='balanced', n_jobs=2),
    'Linear SVM': CalibratedClassifierCV(
        LinearSVC(C=0.5, class_weight='balanced', max_iter=500)),
    'MLP Deep NN': MLPClassifier(
        hidden_layer_sizes=(256, 128, 64), activation='relu',
        solver='adam', alpha=0.001, max_iter=100, early_stopping=True)
}

results = {}
for name, model in models.items():
    print(f"  Training {name}...")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    results[name] = {
        'pred':      y_pred,
        'prob':      y_prob,
        'precision': round(precision_score(y_test, y_pred), 4),
        'recall':    round(recall_score(y_test, y_pred), 4),
        'f1':        round(f1_score(y_test, y_pred), 4),
        'auc':       round(auc(fpr, tpr), 4),
    }

print("All models trained!")

# FIGURE BUILDERS
def fig_class_dist():
    counts = df['fraudulent'].value_counts().reset_index()
    counts.columns = ['Class', 'Count']
    counts['Label'] = counts['Class'].map({0: 'Real', 1: 'Fake'})
    fig = px.bar(counts, x='Label', y='Count', color='Label',
        color_discrete_map={'Real': '#2196F3', 'Fake': '#F44336'},
        title='Class Distribution — Real vs Fake Job Postings', text='Count')
    fig.update_traces(textposition='outside')
    fig.update_layout(showlegend=False, template='plotly_dark')
    return fig

def fig_missing():
    missing = df.isnull().mean() * 100
    missing = missing[missing > 0].sort_values(ascending=False).reset_index()
    missing.columns = ['Column', 'Missing %']
    fig = px.bar(missing, x='Missing %', y='Column', orientation='h',
        title='Missing Value Analysis by Column (%)',
        color='Missing %', color_continuous_scale='Reds',
        text=missing['Missing %'].round(1).astype(str) + '%')
    fig.update_traces(textposition='outside')
    fig.update_layout(yaxis={'categoryorder': 'total ascending'}, template='plotly_dark')
    return fig

def fig_binary():
    binary_cols = ['telecommuting', 'has_company_logo', 'has_questions']
    fig = make_subplots(rows=1, cols=3, subplot_titles=binary_cols)
    colors = {'Real': '#2196F3', 'Fake': '#F44336'}
    for i, col in enumerate(binary_cols):
        grouped = df.groupby([col, 'fraudulent']).size().reset_index(name='count')
        grouped['fl'] = grouped['fraudulent'].map({0: 'Real', 1: 'Fake'})
        grouped[col] = grouped[col].astype(str)
        for label, color in colors.items():
            sub = grouped[grouped['fl'] == label]
            fig.add_trace(go.Bar(x=sub[col], y=sub['count'], name=label,
                marker_color=color, showlegend=(i == 0)), row=1, col=i+1)
    fig.update_layout(barmode='group', height=400,
        title_text='Binary Feature Analysis', template='plotly_dark')
    return fig

def fig_employment():
    emp = df.groupby('employment_type')['fraudulent'].mean().reset_index()
    emp.columns = ['Employment Type', 'Fraud Rate']
    emp = emp.sort_values('Fraud Rate', ascending=False).dropna()
    fig = px.bar(emp, x='Employment Type', y='Fraud Rate',
        title='Employment Type vs Fraud Rate',
        color='Fraud Rate', color_continuous_scale='OrRd',
        text=emp['Fraud Rate'].round(3).astype(str))
    fig.update_traces(textposition='outside')
    fig.update_layout(template='plotly_dark')
    return fig

def fig_industry():
    ind = df.groupby('industry')['fraudulent'].mean().reset_index()
    ind.columns = ['Industry', 'Fraud Rate']
    ind = ind.sort_values('Fraud Rate', ascending=False).head(15).dropna()
    fig = px.bar(ind, x='Fraud Rate', y='Industry', orientation='h',
        title='Top 15 Industries by Fraud Rate',
        color='Fraud Rate', color_continuous_scale='Reds',
        text=ind['Fraud Rate'].round(3).astype(str))
    fig.update_traces(textposition='outside')
    fig.update_layout(yaxis={'categoryorder': 'total ascending'}, template='plotly_dark')
    return fig

def fig_text_len():
    fig = make_subplots(rows=1, cols=3,
        subplot_titles=['Description Length', 'Requirements Length', 'Total Text Length'])
    cols_to_plot = ['desc_len', 'req_len', 'text_len']
    colors = {'Real': '#2196F3', 'Fake': '#F44336'}
    for i, col in enumerate(cols_to_plot):
        for label, color in colors.items():
            sub = df[df['label'] == label][col]
            fig.add_trace(go.Histogram(x=sub, name=label,
                marker_color=color, opacity=0.6, showlegend=(i == 0)),
                row=1, col=i+1)
    fig.update_layout(barmode='overlay', height=400,
        title_text='Text Length Distribution — Real vs Fake', template='plotly_dark')
    return fig

def fig_model_compare():
    rows = [{'Model': n, 'Metric': m.capitalize(), 'Score': results[n][m]}
            for n in results for m in ['precision', 'recall', 'f1']]
    mdf = pd.DataFrame(rows)
    fig = px.bar(mdf, x='Model', y='Score', color='Metric', barmode='group',
        title='Model Comparison — Precision, Recall, F1',
        color_discrete_map={'Precision': '#4CAF50', 'Recall': '#2196F3', 'F1': '#FF9800'},
        text=mdf['Score'].astype(str))
    fig.update_traces(textposition='outside')
    fig.update_layout(yaxis_range=[0, 1.15], template='plotly_dark')
    return fig

def fig_roc():
    fig = go.Figure()
    for name in results:
        fpr, tpr, _ = roc_curve(y_test, results[name]['prob'])
        roc_auc = auc(fpr, tpr)
        fig.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines',
            name=f'{name} (AUC={roc_auc:.3f})'))
    fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines',
        line=dict(dash='dash', color='gray'), name='Random'))
    fig.update_layout(title='ROC Curves — All Models',
        xaxis_title='False Positive Rate', yaxis_title='True Positive Rate',
        template='plotly_dark')
    return fig

def fig_confusion(selected_model):
    cm = confusion_matrix(y_test, results[selected_model]['pred'])
    fig = px.imshow(cm, text_auto=True, color_continuous_scale='Blues',
        x=['Real', 'Fake'], y=['Real', 'Fake'],
        title=f'Confusion Matrix — {selected_model}',
        labels=dict(x='Predicted', y='Actual'))
    fig.update_layout(template='plotly_dark')
    return fig

def fig_pr_curves():
    fig = go.Figure()
    for name in results:
        prec, rec, _ = precision_recall_curve(y_test, results[name]['prob'])
        fig.add_trace(go.Scatter(x=rec, y=prec, mode='lines', name=name))
    fig.update_layout(title='Precision-Recall Curves — All Models',
        xaxis_title='Recall', yaxis_title='Precision', template='plotly_dark')
    return fig

def fig_feature_importance():
    lr = models['Logistic Regression']
    feature_names = tfidf.get_feature_names_out()
    coefs = lr.coef_[0][:len(feature_names)]
    top_n = 15
    top_fake_idx = np.argsort(coefs)[-top_n:][::-1]
    top_real_idx = np.argsort(coefs)[:top_n]
    words  = list(feature_names[top_fake_idx]) + list(feature_names[top_real_idx])
    scores = list(coefs[top_fake_idx]) + list(coefs[top_real_idx])
    labels = ['Fake Indicator'] * top_n + ['Real Indicator'] * top_n
    fdf = pd.DataFrame({'Word': words, 'Coefficient': scores, 'Type': labels})
    fig = px.bar(fdf, x='Coefficient', y='Word', color='Type', orientation='h',
        color_discrete_map={'Fake Indicator': '#F44336', 'Real Indicator': '#2196F3'},
        title='Top Discriminative Words — Fake vs Real')
    fig.update_layout(yaxis={'categoryorder': 'total ascending'},
        height=700, template='plotly_dark')
    return fig

# KPI VALUES
best_model = max(results, key=lambda x: results[x]['f1'])
total_jobs  = len(df)
fake_jobs   = int(df['fraudulent'].sum())
best_f1     = results[best_model]['f1']
best_auc    = results[best_model]['auc']

# LAYOUT
app = JupyterDash(__name__, title='Fake Job Detection Dashboard')

CARD = {
    'background': '#1e2130', 'borderRadius': '10px',
    'padding': '20px', 'textAlign': 'center',
    'flex': '1', 'margin': '8px', 'color': 'white'
}
SECTION = {
    'background': '#1e2130', 'borderRadius': '10px',
    'padding': '20px', 'margin': '16px 0'
}
TAB_STYLE         = {'color': 'white', 'backgroundColor': '#1e2130'}
TAB_SELECTED      = {'color': '#2196F3', 'backgroundColor': '#1e2130'}

app.layout = html.Div(
    style={'backgroundColor': '#0e1117', 'padding': '20px',
           'fontFamily': 'Segoe UI, sans-serif'},
    children=[

        # Header
        html.H1("🕵️ Fake Job Detection Dashboard",
            style={'color': 'white', 'textAlign': 'center', 'marginBottom': '4px'}),
        html.P("EMSCAD Dataset  |  4 ML/DL Models  |  11 Visualizations",
            style={'color': '#aaa', 'textAlign': 'center'}),

        # KPI Cards
        html.Div(style={'display': 'flex', 'flexWrap': 'wrap', 'marginTop': '20px'}, children=[
            html.Div([html.H2(f"{total_jobs:,}", style={'color': '#2196F3', 'margin': 0}),
                      html.P("Total Postings",   style={'color': '#aaa', 'margin': 0})], style=CARD),
            html.Div([html.H2(f"{fake_jobs:,}",  style={'color': '#F44336', 'margin': 0}),
                      html.P("Fake Postings",     style={'color': '#aaa', 'margin': 0})], style=CARD),
            html.Div([html.H2(best_model, style={'color': '#4CAF50', 'margin': 0, 'fontSize': '18px'}),
                      html.P("Best Model",        style={'color': '#aaa', 'margin': 0})], style=CARD),
            html.Div([html.H2(f"{best_f1:.3f}",  style={'color': '#FF9800', 'margin': 0}),
                      html.P("Best F1-Score",     style={'color': '#aaa', 'margin': 0})], style=CARD),
            html.Div([html.H2(f"{best_auc:.3f}", style={'color': '#9C27B0', 'margin': 0}),
                      html.P("Best AUC-ROC",      style={'color': '#aaa', 'margin': 0})], style=CARD),
        ]),

        # Tabs
        dcc.Tabs(
            style={'marginTop': '24px'},
            colors={'border': '#333', 'primary': '#2196F3', 'background': '#1e2130'},
            children=[

                # ── EDA ──
                dcc.Tab(label='📊 EDA', style=TAB_STYLE, selected_style=TAB_SELECTED, children=[
                    html.Div(style=SECTION, children=[dcc.Graph(figure=fig_class_dist())]),
                    html.Div(style={'display': 'flex', 'gap': '16px'}, children=[
                        html.Div(style={**SECTION, 'flex': '1'}, children=[dcc.Graph(figure=fig_missing())]),
                        html.Div(style={**SECTION, 'flex': '1'}, children=[dcc.Graph(figure=fig_employment())]),
                    ]),
                    html.Div(style=SECTION, children=[dcc.Graph(figure=fig_binary())]),
                    html.Div(style=SECTION, children=[dcc.Graph(figure=fig_industry())]),
                    html.Div(style=SECTION, children=[dcc.Graph(figure=fig_text_len())]),
                ]),

                # ── Model Performance ──
                dcc.Tab(label='🤖 Model Performance', style=TAB_STYLE, selected_style=TAB_SELECTED, children=[
                    html.Div(style=SECTION, children=[dcc.Graph(figure=fig_model_compare())]),
                    html.Div(style={'display': 'flex', 'gap': '16px'}, children=[
                        html.Div(style={**SECTION, 'flex': '1'}, children=[dcc.Graph(figure=fig_roc())]),
                        html.Div(style={**SECTION, 'flex': '1'}, children=[dcc.Graph(figure=fig_pr_curves())]),
                    ]),
                ]),

                # ── Confusion Matrix ──
                dcc.Tab(label='🔲 Confusion Matrix', style=TAB_STYLE, selected_style=TAB_SELECTED, children=[
                    html.Div(style=SECTION, children=[
                        html.Label("Select Model:", style={'color': 'white', 'fontSize': '16px'}),
                        dcc.Dropdown(
                            id='cm-dropdown',
                            options=[{'label': m, 'value': m} for m in models],
                            value='Linear SVM',
                            style={'width': '300px', 'marginTop': '8px', 'color': 'black'}
                        ),
                        dcc.Graph(id='cm-graph')
                    ])
                ]),

                # ── Feature Importance ──
                dcc.Tab(label='🔍 Feature Importance', style=TAB_STYLE, selected_style=TAB_SELECTED, children=[
                    html.Div(style=SECTION, children=[dcc.Graph(figure=fig_feature_importance())])
                ]),

                # ── Summary Table ──
                dcc.Tab(label='📋 Summary', style=TAB_STYLE, selected_style=TAB_SELECTED, children=[
                    html.Div(style=SECTION, children=[
                        html.H3("Model Performance Summary", style={'color': 'white'}),
                        html.Table(
                            style={'width': '100%', 'borderCollapse': 'collapse', 'color': 'white'},
                            children=[
                                html.Thead(html.Tr([
                                    html.Th(col, style={'padding': '12px',
                                        'borderBottom': '1px solid #444', 'color': '#2196F3'})
                                    for col in ['Model', 'Precision', 'Recall', 'F1-Score', 'AUC']
                                ])),
                                html.Tbody([
                                    html.Tr([
                                        html.Td(n, style={'padding': '12px', 'borderBottom': '1px solid #333'}),
                                        html.Td(results[n]['precision'], style={'padding': '12px', 'borderBottom': '1px solid #333'}),
                                        html.Td(results[n]['recall'],    style={'padding': '12px', 'borderBottom': '1px solid #333'}),
                                        html.Td(results[n]['f1'],
                                            style={'padding': '12px', 'borderBottom': '1px solid #333',
                                                   'color': '#4CAF50' if n == best_model else 'white',
                                                   'fontWeight': 'bold' if n == best_model else 'normal'}),
                                        html.Td(results[n]['auc'],       style={'padding': '12px', 'borderBottom': '1px solid #333'}),
                                    ]) for n in results
                                ])
                            ]
                        )
                    ])
                ]),
            ]
        ),
    ]
)

# CALLBACK
@app.callback(
    Output('cm-graph', 'figure'),
    Input('cm-dropdown', 'value')
)
def update_confusion_matrix(selected_model):
    return fig_confusion(selected_model)

# RUN
if __name__ == '__main__':
    app.run(debug=True, jupyter_mode='external')

Training models... please wait.
  Training Logistic Regression...
  Training Random Forest...
  Training Linear SVM...
  Training MLP Deep NN...
All models trained!
Dash app running on http://127.0.0.1:8050/
